# 📦 InklusiKerja — Step 1: Data Preprocessing Pipeline (v2)

Pipeline ini memproses data mentah pekerjaan dan kandidat untuk sistem rekomendasi InklusiKerja.

**Yang dilakukan:**
- Normalisasi skill ambigu (`r` → `r programming`, `go` → `golang`)
- Enrichment skill tags per job title (JOB_SKILL_MAP)
- Membangun kolom `document_text` untuk jobs (diperkaya skill)
- Membangun kolom `query_text` untuk kandidat

**Output:** `data/processed/jobs_processed.csv`, `data/processed/kandidat_processed.csv`, `data/processed/data_report.json`

## 1. Import & Konfigurasi

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os

RAW_JOBS_CSV     = "data/job_titles_disabilitas.csv"
RAW_KANDIDAT_CSV = "data/kandidat_dummy.csv"
OUTPUT_DIR       = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Konfigurasi selesai")
print(f"   Jobs CSV    : {RAW_JOBS_CSV}")
print(f"   Kandidat CSV: {RAW_KANDIDAT_CSV}")
print(f"   Output Dir  : {OUTPUT_DIR}")

## 2. Definisi Skill Normalization & Known Skills

In [ ]:
# Skill alias yang ambigu → nama lengkap
SKILL_NORMALIZE = {
    "r"  : "r programming",
    "go" : "golang",
}

# Kumpulan skill yang dikenal sistem
KNOWN_SKILLS = {
    "python", "r programming", "golang", "java", "javascript", "typescript",
    "php", "kotlin", "swift", "dart", "css", "html",
    "react", "vue.js", "node.js", "django", "fastapi", "spring boot",
    "laravel", "flutter", "react native", "bootstrap", "jetpack compose",
    "redis", "microservices", "rest api",
    "power bi", "tableau", "google data studio", "numpy", "pandas",
    "sql", "mysql", "postgresql", "sqlite", "excel", "google sheets",
    "machine learning dasar", "statistik", "data cleaning",
    "docker", "kubernetes", "aws", "gcp", "azure", "linux", "ci/cd",
    "jenkins", "terraform", "ansible",
    "figma", "adobe xd", "sketch", "invision", "canva",
    "adobe illustrator", "adobe photoshop", "coreldraw",
    "wireframing", "prototyping", "user research", "design system",
    "seo", "google ads", "facebook ads", "tiktok ads",
    "email marketing", "copywriting", "content creation",
    "digital marketing", "analitik media sosial",
    "hris", "sap", "erp", "myob", "accurate",
    "firewall", "owasp", "penetration testing", "metasploit",
    "ethical hacking", "kriptografi", "iso 27001", "siem", "nmap",
    "selenium", "cypress", "katalon", "postman", "jira",
    "qa documentation", "test case", "testing manual", "bug tracking",
    "cisco", "mikrotik", "vpn", "ccna",
    "monitoring jaringan", "server management", "networking",
    "microsoft office", "microsoft excel", "microsoft word",
    "google workspace", "administrasi perkantoran",
    "zendesk", "freshdesk", "crm", "live chat",
    "komunikasi", "problem solving", "empati", "penanganan keluhan",
    "notulensi", "manajemen jadwal", "pengarsipan",
    "wms", "sap wm", "inventory", "supply chain",
    "rekrutmen", "payroll", "manajemen kinerja",
    "pelatihan sdm", "hubungan industrial", "linkedin recruiter",
    "e-learning", "zoom", "google classroom", "moodle",
    "microsoft teams", "kurikulum",
    "motion graphics", "tipografi", "desain visual", "branding",
    "editing video", "capcut", "youtube", "instagram",
    "penulisan konten", "seo writing", "wordpress",
    "akuntansi", "jurnal keuangan", "pajak", "rekonsiliasi bank",
    "laporan keuangan", "audit internal",
    "manajemen gudang", "perencanaan distribusi",
    "penerjemahan", "proofreading", "lokalisasi", "sdl trados",
    "bahasa inggris", "bahasa jepang", "bahasa mandarin",
    "ketelitian data", "pengetikan cepat", "spreadsheet",
}

print(f"✅ KNOWN_SKILLS: {len(KNOWN_SKILLS)} skill terdaftar")

## 3. JOB_SKILL_MAP — Canonical Skill Pool per Job Title

In [ ]:
JOB_SKILL_MAP = {
    "Customer Service Representative" : ["komunikasi", "problem solving", "crm", "penanganan keluhan", "empati"],
    "Customer Support Agent"          : ["zendesk", "freshdesk", "komunikasi", "live chat", "penanganan keluhan"],
    "Customer Support Specialist"     : ["crm", "komunikasi", "problem solving", "live chat", "empati"],
    "Telemarketer"                    : ["komunikasi", "crm", "problem solving"],
    "Telesales Agent"                 : ["komunikasi", "crm", "problem solving"],
    "Data Analyst"                    : ["sql", "excel", "python", "tableau", "power bi", "statistik", "pandas"],
    "Data Entry Operator"             : ["microsoft excel", "ketelitian data", "pengetikan cepat", "accurate", "spreadsheet"],
    "Data Entry Specialist"           : ["microsoft excel", "ketelitian data", "pengetikan cepat", "accurate", "spreadsheet"],
    "Graphic Designer"                : ["adobe illustrator", "adobe photoshop", "canva", "coreldraw", "desain visual", "tipografi"],
    "Illustrator"                     : ["adobe illustrator", "adobe photoshop", "coreldraw", "desain visual"],
    "Animator"                        : ["motion graphics", "canva", "adobe illustrator", "adobe photoshop", "tipografi"],
    "UI/UX Designer"                  : ["figma", "adobe xd", "wireframing", "prototyping", "user research", "design system", "sketch", "invision"],
    "Software Developer"              : ["python", "java", "javascript", "typescript", "react", "node.js", "docker", "git", "rest api"],
    "Programmer"                      : ["python", "java", "javascript", "php", "git", "rest api"],
    "QA Tester"                       : ["selenium", "cypress", "katalon", "postman", "jira", "qa documentation", "test case", "bug tracking", "testing manual"],
    "Digital Marketing Specialist"    : ["seo", "google ads", "facebook ads", "tiktok ads", "email marketing", "copywriting", "content creation", "analitik media sosial"],
    "Social Media Manager"            : ["instagram", "analitik media sosial", "copywriting", "content creation", "canva"],
    "Content Writer"                  : ["copywriting", "seo writing", "penulisan konten", "wordpress", "instagram"],
    "Freelance Writer"                : ["copywriting", "penulisan konten", "seo writing", "proofreading"],
    "Translator"                      : ["penerjemahan", "bahasa inggris", "bahasa jepang", "bahasa mandarin", "proofreading", "lokalisasi", "sdl trados"],
    "HR Specialist (Remote)"          : ["hris", "rekrutmen", "payroll", "manajemen kinerja", "pelatihan sdm", "linkedin recruiter", "hubungan industrial"],
    "Accountant"                      : ["akuntansi", "jurnal keuangan", "pajak", "accurate", "sap", "myob", "rekonsiliasi bank"],
    "Bookkeeper"                      : ["jurnal keuangan", "akuntansi", "accurate", "microsoft excel", "rekonsiliasi bank"],
    "Financial Analyst"               : ["microsoft excel", "sql", "python", "tableau", "laporan keuangan", "statistik"],
    "Administrative Clerk"            : ["microsoft office", "administrasi perkantoran", "manajemen jadwal", "notulensi", "google workspace"],
    "Virtual Assistant"               : ["microsoft office", "google workspace", "manajemen jadwal", "administrasi perkantoran", "notulensi"],
    "Remote Project Manager"          : ["microsoft office", "google workspace", "manajemen jadwal", "jira"],
    "Online Tutor"                    : ["e-learning", "zoom", "google classroom", "moodle", "microsoft teams", "kurikulum"],
    "Video Editor"                    : ["capcut", "editing video", "adobe photoshop", "youtube"],
    "Podcast Producer"                : ["editing video", "copywriting"],
    "Voice-over Artist"               : ["komunikasi"],
    "Transcriptionist"                : ["pengetikan cepat", "ketelitian data", "microsoft word"],
    "Researcher"                      : ["statistik", "python", "r programming", "google sheets", "microsoft excel"],
    "E-commerce Manager"              : ["digital marketing", "microsoft excel", "seo", "content creation"],
    "Online Moderator"                : ["komunikasi", "microsoft office", "google workspace"],
    "Archivist"                       : ["pengarsipan", "administrasi perkantoran", "microsoft office", "ketelitian data"],
    "Library Assistant"               : ["pengarsipan", "administrasi perkantoran", "microsoft office"],
    "Retail Stock Assistant"          : ["inventory", "microsoft excel", "manajemen gudang"],
    "Garden Maintenance Worker"       : [],
    "Laundry Worker"                  : [],
    "Food Packaging Worker"           : [],
    "Craft/Artisan Worker"            : [],
    "Cleaning Service"                : [],
    "Horticultural Therapist Assistant": [],
    "Art Therapist Assistant"         : [],
}

print(f"✅ JOB_SKILL_MAP: {len(JOB_SKILL_MAP)} job titles terdaftar")

## 4. Helper Functions

In [ ]:
def clean_text(text: str) -> str:
    """Lowercase, hapus karakter aneh, normalisasi spasi."""
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s,./\-+()]", "", text)
    return text


def normalize_skill(s: str) -> str:
    """Normalisasi satu skill: strip + lowercase + ganti alias ambigu."""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return SKILL_NORMALIZE.get(s, s)


def extract_skills_from_text(text: str) -> list:
    """Cari skill yang dikenal (KNOWN_SKILLS) dari teks deskripsi."""
    text_lower = text.lower()
    found = []
    for skill in KNOWN_SKILLS:
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, text_lower):
            found.append(skill)
    return found


def get_job_skill_tags(row: pd.Series) -> list:
    """Gabungkan skill dari deskripsi kualifikasi + JOB_SKILL_MAP."""
    extracted = extract_skills_from_text(row["Deskripsi Kualifikasi"])
    mapped    = JOB_SKILL_MAP.get(row["Job Title"], [])
    combined  = list(dict.fromkeys(extracted + mapped))  # deduplicate
    return combined


def build_job_document(row: pd.Series) -> str:
    """Buat document_text untuk embedding jobs (diperkaya skill)."""
    skill_part = ""
    if row["skill_tags"]:
        skill_part = f" | skill relevan {', '.join(row['skill_tags'])}"
    parts = [
        f"posisi {clean_text(row['Job Title'])}",
        f"jenis disabilitas {clean_text(row['Jenis Disabilitas'])}",
        f"level {clean_text(row['Level'])}",
        f"kualifikasi {clean_text(row['Deskripsi Kualifikasi'])}",
    ]
    doc = " | ".join(parts)
    doc += skill_part
    doc += f" | aksesibilitas yang disediakan {clean_text(row['Kebutuhan Aksesibilitas'])}"
    return doc


def build_kandidat_document(row: pd.Series) -> str:
    """Buat query_text untuk embedding kandidat."""
    skills_str = ", ".join(row["skills_list"])
    parts = [
        f"jenis disabilitas {clean_text(row['disability_type'])}",
        f"skill yang dimiliki {skills_str}",
        f"profil fungsional {clean_text(row['functional_profile'])}",
    ]
    return " | ".join(parts)


print("✅ Helper functions siap")

## 5. Processing Jobs

In [ ]:
print("📂 Membaca data pekerjaan...")
jobs_df = pd.read_csv(RAW_JOBS_CSV)
print(f"   → {len(jobs_df)} baris ditemukan")
print(f"   → Kolom: {jobs_df.columns.tolist()}")

# Deduplikasi
before = len(jobs_df)
jobs_df = jobs_df.drop_duplicates()
print(f"   → Setelah dedup: {len(jobs_df)} baris ({before - len(jobs_df)} dihapus)")

jobs_df = jobs_df.reset_index(drop=True)
jobs_df["job_id"] = jobs_df.index.map(lambda i: f"JOB{i:04d}")

# Skill tags enrichment
jobs_df["skill_tags"] = jobs_df.apply(get_job_skill_tags, axis=1)

# Document text
jobs_df["document_text"] = jobs_df.apply(build_job_document, axis=1)

# Level ranking
level_order = {"entry level": 1, "mid level": 2, "senior level": 3, "lead / manajerial": 4}
jobs_df["level_rank"] = jobs_df["Level"].str.lower().map(level_order).fillna(2)

print(f"\n📊 Distribusi Jenis Disabilitas:")
print(jobs_df["Jenis Disabilitas"].value_counts())
print(f"\n📊 Distribusi Level:")
print(jobs_df["Level"].value_counts())

jobs_with_skills = jobs_df[jobs_df["skill_tags"].str.len() > 0]
print(f"\n📊 Jobs dengan skill tags: {len(jobs_with_skills)}/{len(jobs_df)}")

jobs_df.head(3)

In [ ]:
# Simpan jobs
out_path = f"{OUTPUT_DIR}/jobs_processed.csv"
jobs_df.to_csv(out_path, index=False)
print(f"✅ Jobs saved → {out_path}")
print(f"\nSample document_text:")
print(jobs_df['document_text'].iloc[0])

## 6. Processing Kandidat

In [ ]:
print("📂 Membaca data kandidat...")
kandidat_df = pd.read_csv(RAW_KANDIDAT_CSV)
print(f"   → {len(kandidat_df)} kandidat ditemukan")

def parse_and_normalize(skills_raw: str) -> list:
    skills = [s.strip() for s in str(skills_raw).split(",") if s.strip()]
    normalized = [normalize_skill(s) for s in skills]
    seen, result = set(), []
    for s in normalized:
        if s not in seen:
            seen.add(s)
            result.append(s)
    return result

kandidat_df["skills_list"] = kandidat_df["skills"].apply(parse_and_normalize)

def to_display(skills_list: list) -> str:
    display_map = {"r programming": "R Programming", "golang": "Golang"}
    return ", ".join(display_map.get(s, s.title()) for s in skills_list)

kandidat_df["skills"] = kandidat_df["skills_list"].apply(to_display)
kandidat_df["query_text"] = kandidat_df.apply(build_kandidat_document, axis=1)

print(f"\n📊 Distribusi Jenis Disabilitas Kandidat:")
print(kandidat_df["disability_type"].value_counts())

all_skills = [s for lst in kandidat_df["skills_list"] for s in lst]
normalized_count = sum(1 for s in all_skills if s in ("r programming", "golang"))
print(f"\n✅ Skill ternormalisasi (r/go → nama lengkap): {normalized_count} entri")

kandidat_df.head(3)

In [ ]:
# Simpan kandidat
out_path = f"{OUTPUT_DIR}/kandidat_processed.csv"
kandidat_df.to_csv(out_path, index=False)
print(f"✅ Kandidat saved → {out_path}")
print(f"\nSample query_text:")
print(kandidat_df['query_text'].iloc[0])

## 7. Generate Data Report

In [ ]:
report = {
    "jobs": {
        "total_rows"          : len(jobs_df),
        "unique_job_titles"   : jobs_df["Job Title"].nunique(),
        "disability_types"    : jobs_df["Jenis Disabilitas"].nunique(),
        "levels"              : jobs_df["Level"].unique().tolist(),
        "jobs_with_skill_tags": int((jobs_df["skill_tags"].str.len() > 0).sum()),
        "sample_document"     : jobs_df["document_text"].iloc[0],
    },
    "kandidat": {
        "total_rows"      : len(kandidat_df),
        "disability_types": kandidat_df["disability_type"].nunique(),
        "sample_query"    : kandidat_df["query_text"].iloc[0],
    }
}

report_path = f"{OUTPUT_DIR}/data_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"📋 Data report → {report_path}")
print("\n━━━ SAMPLE JOB DOCUMENT ━━━")
print(jobs_df["document_text"].iloc[0])
print("\n━━━ SAMPLE KANDIDAT QUERY ━━━")
print(kandidat_df["query_text"].iloc[0])
print("\n✅ Preprocessing selesai! Lanjut ke 02_embedding.ipynb")